Perfect 👍 I’ll update your scaffold so that the **Parent Agent** now performs **LLM-based routing** using the model defined in `config/settings.py`.

Here’s the **fully updated code**:

---

# 📂 Project Structure

```
order_mgmt_framework/
│── main.py
│
├── config/
│   ├── __init__.py
│   ├── settings.py
│
├── mcp/
│   ├── __init__.py
│   ├── server_weather.py
│   ├── client_weather.py
│   ├── server_pollution.py
│   └── client_pollution.py
│
├── tools/
│   ├── __init__.py
│   ├── weather_tools.py
│   └── pollution_tools.py
│
├── agents/
│   ├── __init__.py
│   ├── agent_factory.py
│   └── parent_agent.py
```

---

# ⚙️ `config/settings.py`

```python
# config/settings.py

AGENT_CONFIG = {
    "weather": {
        "llm": "openai",
        "tools": ["get_city_weather", "get_country_weather"],
        "mcp_servers": ["weather-mcp"]
    },
    "pollution": {
        "llm": "openai",
        "tools": ["get_city_pollution", "get_country_pollution"],
        "mcp_servers": ["pollution-mcp"]
    },
    "parent": {
        "llm": {
            "provider": "openai",       # can switch to "gemini", "anthropic", etc.
            "model": "gpt-4o-mini"      # configurable LLM model
        },
        "agents": ["weather", "pollution"],
        "protocol": "A2A",
        "features": ["RAG", "React", "ChainOfThought"]
    }
}
```

---

# 🏭 `agents/parent_agent.py` (LLM Router)

```python
# agents/parent_agent.py
from config.settings import AGENT_CONFIG
from openai import OpenAI

def get_llm_client():
    llm_conf = AGENT_CONFIG["parent"]["llm"]
    provider = llm_conf.get("provider", "openai")

    if provider == "openai":
        return OpenAI(), llm_conf["model"]

    # 🔧 Future: extend here for Gemini/Anthropic
    raise ValueError(f"Unsupported LLM provider: {provider}")

def llm_router(prompt: str) -> str:
    """
    Use the configured LLM to decide which agent to call (weather or pollution).
    """
    client, model = get_llm_client()

    routing_prompt = f"""
    You are a router. Decide whether this query is about weather, pollution, or none.

    Query: "{prompt}"

    Respond with only one word: "weather", "pollution", or "end".
    """

    resp = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": routing_prompt}],
        max_tokens=1,
    )

    decision = resp.choices[0].message.content.strip().lower()
    if decision not in ["weather", "pollution", "end"]:
        return "end"
    return decision
```

---

# 🚀 `main.py`

```python
# main.py
import asyncio
from langgraph.graph import StateGraph, START, END
from agents.agent_factory import AgentFactory
from agents.parent_agent import llm_router

def build_graph():
    factory = AgentFactory()
    weather_agent = factory.get_agent("weather")
    pollution_agent = factory.get_agent("pollution")

    builder = StateGraph(dict)

    # Parent router node
    builder.add_node("parent_router", lambda state: {"next": llm_router(state["prompt"])})

    # Weather agent node
    builder.add_node("weather", lambda state: {"result": asyncio.run(weather_agent(state))})

    # Pollution agent node
    builder.add_node("pollution", lambda state: {"result": asyncio.run(pollution_agent(state))})

    # Graph wiring
    builder.add_edge(START, "parent_router")
    builder.add_conditional_edges(
        "parent_router",
        lambda out: out["next"],
        {"weather": "weather", "pollution": "pollution", "end": END}
    )
    builder.add_edge("weather", END)
    builder.add_edge("pollution", END)

    return builder.compile()

def main():
    graph = build_graph()

    prompts = [
        {"prompt": "What's the weather in Paris?", "city": "Paris"},
        {"prompt": "Check AQI in Delhi", "city": "Delhi"},
        {"prompt": "Tell me a joke"}  # should route to "end"
    ]

    for p in prompts:
        print(f"\n📝 Prompt: {p['prompt']}")
        result = graph.invoke(p)
        print("🤖 Response:", result.get("result"))

if __name__ == "__main__":
    main()
```

---

# ✅ Key Upgrades

* **Parent agent LLM config** lives in `settings.py`.
* **Router model** is **not hardcoded** — you can switch provider/model in config.
* Parent uses `langgraph` state machine to direct flow.
* Weather & pollution MCP servers stay the same.

---

👉 Do you also want me to make the **parent itself an MCP server** (so you can call `ask_parent` and it internally routes to child MCP agents)? That would let you expose the whole framework over MCP.
